In [0]:
movie_df = (
    spark.read
    .table("workspace.default.movie_genre_silver")
)

display(movie_df)

In [0]:
ratings_df = (
    spark.read
    .table("workspace.default.ratings")
)

display(ratings_df)

In [0]:
user_df = (
    spark.read
    .table("workspace.default.users")
)

display(user_df)

In [0]:
occupation_df = (
    spark.read
    .table("workspace.default.occupations")
)

In [0]:
from pyspark.sql import functions as F

user_stats_df = (
    ratings_df
    .groupBy("userId")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("rating").alias("num_ratings"),
    )
)

# Join user stats with user info to get age, gender, and occupation
user_stats_df = (
    user_stats_df
    .join(user_df, on="userId", how="left")
    .join(
        occupation_df,
        user_stats_df["occupation"] == F.col("occupation_id").cast("string"),
        how="left",
    )
    .select(
        "userId",
        "avg_rating",
        "num_ratings",
        "age",
        "gender",
        F.col("occupation_name").alias("occupation"),
    )
)

display(user_stats_df)

In [0]:
(
    user_stats_df.write
    .mode("overwrite")
    .saveAsTable("workspace.default.user_stats_gold")
)